<!-- @format -->

# Kaggle: Trích xuất vector đặc trưng → file `.npz`

Notebook này chỉ chạy **feature extraction** (GPU), lưu cache giống `main.py --save-feature-cache`: **features RAW, không PCA** trong file `.npz`. Sau khi tải về máy, đặt vào `results/cache/` và chạy `python main.py --use-feature-cache` để graph + clustering + PCA đúng pipeline.

## Trước khi chạy

1. **Settings → Accelerator:** GPU (T4 / P100).
2. **Settings → Internet:** Bật — cần để `torch.hub` tải DINOv2 lần đầu.
3. **Add Data:** gắn dataset parquet (**ImageNet-Hard** hoặc tên bạn đã upload).
4. **Code dự án:** một trong các cách:
    - `git clone` repo **public** vào `/kaggle/working/`, hoặc
    - Zip thư mục repo (có `src/`, `scripts/`), upload thêm một Kaggle Dataset → Add Data → giải zip vào `/kaggle/working/`.

**Lưu ý đường dẫn:** dataset trên Kaggle mount tại `/kaggle/input/<slug>/`. Slug thường **chữ thường + gạch** (vd `imagenet-hard`), không nhất thiết trùng tên hiển thị. Chạy cell khám phá bên dưới để copy đúng đường dẫn tới thư mục chứa `*.parquet`.

## Sau khi chạy xong

- Tab **Output** của notebook (hoặc file trong `/kaggle/working/`) → download file `.npz`.
- Local: copy vào `D:\...\results\cache\features_<model>_full.npz`, chỉnh `MODEL_NAME` trong `src/config.py` khớp model đã extract.
- Chạy: `python main.py --use-feature-cache`.


In [ ]:
# Cell 1 — Kiểm tra GPU và PyTorch
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


<!-- @format -->

## Khám phá `/kaggle/input` và tìm thư mục chứa `.parquet`

`load_data(path)` đọc mọi file `*.parquet` **trực tiếp** trong `path`. Nếu parquet nằm trong subfolder, đặt `DATA_PATH_KAGGLE` trỏ vào đúng subfolder đó.


In [ ]:
import glob
import os

INPUT_ROOT = "/kaggle/input"
print("Datasets:", os.listdir(INPUT_ROOT))

for root, dirs, files in os.walk(INPUT_ROOT):
    pq = [f for f in files if f.endswith(".parquet")]
    if pq:
        print("PARQUET DIR:", root, "->", pq[:3], "..." if len(pq) > 3 else "")

# Gán sau khi đã xác định đúng thư mục (vd một dòng PARQUET DIR ở trên):
DATA_PATH_KAGGLE = "/kaggle/input/sdfsdf/imagenet-hard"  # USER: sửa cho khớp máy bạn
assert glob.glob(os.path.join(DATA_PATH_KAGGLE, "*.parquet")), f"Không thấy *.parquet trong {DATA_PATH_KAGGLE}"
print("OK DATA_PATH_KAGGLE =", DATA_PATH_KAGGLE)


<!-- @format -->

## Cài dependency tối thiểu + chạy script extract

- Chỉnh `PROJECT_ROOT`: thư mục repo có `src/` và `scripts/extract_features_only.py`.
- `OUTPUT_NPZ`: nên để trong `/kaggle/working/` để dễ download (tab Output).
- Có thể chỉnh `MODEL_NAME` / `BATCH_SIZE` trong `src/config.py` trên repo trước khi zip/clone, hoặc truyền `--model-name` (xem cell sau).


In [ ]:
!pip install -q tqdm scipy


In [ ]:
# Cell — USER: chỉnh PROJECT_ROOT sau khi clone hoặc giải zip code vào /kaggle/working
import os
import subprocess
import sys

PROJECT_ROOT = "/kaggle/working/DADN_TTNT"  # USER: đường dẫn tới thư mục chứa src/
OUTPUT_NPZ = "/kaggle/working/features_dinov2_vitb14_full.npz"  # USER: đổi tên model nếu cần

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src")), (
    f"Không thấy {PROJECT_ROOT}/src — hãy clone hoặc giải zip repo và cập nhật PROJECT_ROOT"
)

script = os.path.join(PROJECT_ROOT, "scripts", "extract_features_only.py")
assert os.path.isfile(script), f"Thiếu file {script}"

cmd = [
    sys.executable,
    script,
    "--data-path",
    DATA_PATH_KAGGLE,
    "--output",
    OUTPUT_NPZ,
    "--project-root",
    PROJECT_ROOT,
    "--device",
    "cuda",
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd, cwd=PROJECT_ROOT)
print("Done.")


<!-- @format -->

### Tuỳ chọn: ghi đè model khác (không sửa `config.py`)

Uncomment và chạy lại cell trên với tham số `--model-name dinov2_vitl14` (hoặc chạy lệnh dưới).


In [ ]:
# Tuỳ chọn: extract với model khác (ví dụ ViT-L)
# import subprocess, sys, os
# OUT = "/kaggle/working/features_dinov2_vitl14_full.npz"
# subprocess.check_call([
#     sys.executable,
#     os.path.join(PROJECT_ROOT, "scripts", "extract_features_only.py"),
#     "--data-path", DATA_PATH_KAGGLE,
#     "--output", OUT,
#     "--project-root", PROJECT_ROOT,
#     "--device", "cuda",
#     "--model-name", "dinov2_vitl14",
# ], cwd=PROJECT_ROOT)


<!-- @format -->

## Kiểm tra file `.npz` trước khi download


In [ ]:
import json
import numpy as np

out = globals().get("OUTPUT_NPZ", "/kaggle/working/features_dinov2_vitb14_full.npz")
z = np.load(out, allow_pickle=True)
print("keys:", list(z.files))
print("features:", z["features"].shape, z["features"].dtype)
if "metadata" in z.files:
    mr = z["metadata"].item()
    mr = mr.decode("utf-8") if isinstance(mr, bytes) else str(mr)
    print("metadata:", json.loads(mr))
